In [ ]:
import os
import re
import shutil
import pandas as pd

# ==================== 配置路径 ====================
EXCEL_PATH = "现场工作阶段.xlsx"     # 你的 Excel 清单文件路径
SRC_FOLDER = r"D:\\files\\in_field"     # 存放你目前所有扁平、未分类 Word 文件的文件夹
TARGET_ROOT = r"D:\\files\\现场工作阶段"   # 自动生成的树状多级文件夹根目录
# ==================================================

In [ ]:
def clean_foldername(name):
    """清除文件夹名称中的换行符、Windows非法字符及前后空白"""
    if pd.isna(name):
        return ""
    # 1. 强转为字符串
    name_str = str(name)
    # 2. 核心修复：剔除文本内部可能藏匿的 换行符(\n)、回车符(\r)、制表符(\t)
    name_str = re.sub(r'[\n\r\t]', '', name_str)
    # 3. 替换 Windows 系统标准的非法字符 \ / : * ? " < > |
    name_str = re.sub(r'[\\/:*?"<>|]', '_', name_str)
    # 4. 去除首尾空格
    return name_str.strip()

def get_level(code):
    """
    根据 Excel 的索引号判断当前行属于第几级目录（从 0 开始）
    第三部分 -> Level 0
    第一章   -> Level 1
    1-1      -> Level 2
    1-1-1    -> Level 3
    1-1-4-1  -> Level 4
    """
    code = str(code).strip()
    if '部分' in code:
        return 0
    if '章' in code:
        return 1
    # 匹配数字和短横线的组合，如 1-1, 1-1-1
    if re.match(r'^[\d-]+$', code):
        return len(code.split('-'))
    return None

def build_structure_and_move():
    if not os.path.exists(EXCEL_PATH):
        print(f"❌ 未找到Excel清单文件：{EXCEL_PATH}，请检查路径。")
        return
        
    # 1. 读取Excel（默认第一列为编号，第二列为名称）
    df = pd.read_excel(EXCEL_PATH)
    code_col = df.columns[0]
    name_col = df.columns[1]
    
    dir_mapping = {}      # 存放 编号 -> 文件夹绝对路径 的映射
    current_paths = {-1: TARGET_ROOT}  # 追踪当前各层级的最新文件夹路径
    
    print("📂 正在解析 Excel 并构建多级树状文件夹结构...")
    for _, row in df.iterrows():
        # 对索引号 code 也进行去换行清洗，防止索引号里带换行
        code = re.sub(r'[\n\r\t]', '', str(row[code_col])).strip()
        name = clean_foldername(row[name_col])
        
        if not code or code == "nan" or name == "nan":
            continue
            
        level = get_level(code)
        if level is None:
            continue  # 无法识别的编码格式则跳过
            
        # 寻找当前层级的父目录路径（向上寻找最近的一个有效上级层级）
        parent_level = level - 1
        while parent_level >= -1 and parent_level not in current_paths:
            parent_level -= 1
            
        parent_path = current_paths[parent_level]
        
        # 拼接并创建当前文件夹
        dir_name = f"{code} {name}".strip()
        sub_dir = os.path.join(parent_path, dir_name)
        
        try:
            os.makedirs(sub_dir, exist_ok=True)
        except Exception as e:
            print(f"⚠️ 创建文件夹失败，跳过该行。路径: {sub_dir}，错误: {e}")
            continue
        
        # 更新当前层级及映射关系
        current_paths[level] = sub_dir
        dir_mapping[code] = sub_dir
        
        # 清理比当前层级更深的旧追踪记录（防止跨大类时发生层级错乱）
        expired_levels = [l for l in current_paths if l > level]
        for l in expired_levels:
            del current_paths[l]

    print("✅ 文件夹树状结构创建完毕！开始自动归类 Word 文件...")

    # 2. 扫描源文件夹中的 Word 并通过前缀编号分发
    if not os.path.exists(SRC_FOLDER):
        print(f"⚠️ 存放未分类文件的源文件夹 {SRC_FOLDER} 不存在，请先创建并放入文件。")
        return
        
    files = [f for f in os.listdir(SRC_FOLDER) if f.endswith('.docx') and not f.startswith('~$')]
    move_count = 0
    
    for filename in files:
        # 正则匹配文件名前面的数字编号（支持多组连字符，如 9-2-2-1-1）
        match = re.match(r'^([\d-]+)', filename.strip())
        if match:
            file_code = match.group(1)
            target_dir = None
            
            # 逐级向上递归查找父级文件夹
            parts = file_code.split('-')
            for i in range(len(parts), 0, -1):
                current_code = "-".join(parts[:i])
                if current_code in dir_mapping:
                    target_dir = dir_mapping[current_code]
                    break
            
            if target_dir:
                src_file = os.path.join(SRC_FOLDER, filename)
                dst_file = os.path.join(target_dir, filename)
                try:
                    shutil.move(src_file, dst_file)
                    print(f"[归类成功] {filename} \n      -> {target_dir.replace(TARGET_ROOT, '')}")
                    move_count += 1
                except Exception as e:
                    print(f"[移动失败] {filename}，错误: {e}")
            else:
                print(f"[无法归类] 未在Excel中找到文件编号对应的任何层级目录: {filename}")
        else:
            print(f"[跳过] 文件开头不含标准数字编号: {filename}")
            
    print(f"\n归类任务完成！共成功搬运了 {move_count} 个文件。")

if __name__ == "__main__":
    build_structure_and_move()